In [ ]:
import numpy as np
from tensorflow import keras

INPUT_SHAPE = 10 #how many frames we use to make a predict
NUM_FEATURES = 21*8
NUM_CLASSES = 1001 #1000 + тишина

model = keras.Sequential(
    keras.layers.GRU(256, return_sequences=True, input_shape = (INPUT_SHAPE, NUM_FEATURES),
                     dropout = 0.25, recurrent_dropout = 0.2),
    keras.layers.GRU(128, return_sequences=False,
                    dropout=0.25, recurrent_dropout=0.2),
    keras.layers.GRU(64, return_sequences=False,
                     dropout=0.25, recurrent_dropout=0.2),

    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(NUM_CLASSES)
)

model.compile(optimizer='adam',
              loss='sparse_categorical',
              metrics=['accuracy'])



In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import time

class GestureRecognizer:
    def __init__(self, model=model):
        self.hands = mp.solutions.hands.Hands()
        self.model = model
        self.landmarks_buffer = []
        self.prev_landmarks = None
        self.last_prediction_time = 0

    def extract_landmarks(self, frame):
        results = self.hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        
        if not results.multi_hand_landmarks:
            self.prev_landmarks = None
            return [0.0] * 168, frame
            
        hand_landmarks = results.multi_hand_landmarks[0]
        current_coords = []
        
        for landmark in hand_landmarks.landmark:
            current_coords.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
        
        if self.prev_landmarks is None:
            self.prev_landmarks = current_coords
            return current_coords + [0.0] * 84, frame
            
        changes = [curr - prev for curr, prev in zip(current_coords, self.prev_landmarks)]
        self.prev_landmarks = current_coords
        return current_coords + changes, frame

    def predict_online(self, frames_for_prediction=10, prediction_interval=0.1):
        cap = cv2.VideoCapture(0)
        
        while True:
            success, frame = cap.read()
            if not success:
                break
            
            current_time = time.time()
            landmarks, _ = self.extract_landmarks(frame)
            self.landmarks_buffer.append(landmarks)
            
            if (current_time - self.last_prediction_time >= prediction_interval and 
                len(self.landmarks_buffer) >= frames_for_prediction):
                
                sequence = np.array(self.landmarks_buffer[-frames_for_prediction:])
                prediction = self.model.predict(sequence.reshape(1, frames_for_prediction, -1), verbose=0)
                
                if prediction is not None:
                    predicted_class = np.argmax(prediction[0])
                    confidence = np.max(prediction[0])
                    print(f"Prediction: Class {predicted_class}, Confidence: {confidence:.2f}")
                
                self.last_prediction_time = current_time
                self.landmarks_buffer = self.landmarks_buffer[5:]
            
            cv2.imshow('Gesture Recognition', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
import matplotlib.pyplot as plt

class Visualize:
    
    def plot_training_history(history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        # Accuracy
        ax1.plot(history.history['accuracy'], label='Training Accuracy')
        ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
        ax1.set_title('Model Accuracy')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        
        # Loss
        ax2.plot(history.history['loss'], label='Training Loss')
        ax2.plot(history.history['val_loss'], label='Validation Loss')
        ax2.set_title('Model Loss')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()


    def plot_cm_with_matplotlib(y_true, y_pred, class_names):
        cm = plt.confusion_matrix(y_true, y_pred)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
        ax.figure.colorbar(im, ax=ax)
        
        # Подписи осей
        ax.set(xticks=np.arange(cm.shape[1]),
               yticks=np.arange(cm.shape[0]),
               xticklabels=class_names,
               yticklabels=class_names,
               title='Confusion Matrix',
               ylabel='True Label',
               xlabel='Predicted Label')
        
        # Поворот подписей
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        
        # Добавление чисел в ячейки
        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, format(cm[i, j], 'd'),
                       ha="center", va="center",
                       color="white" if cm[i, j] > thresh else "black")
        
        plt.tight_layout()
        plt.show()
        
        return cm

In [ ]:
from sklearn.model_selection import train_test_split

data = np.load('landmarks.npy', allow_pickle=True).item()
landmarks = data['landmarks']
target = data['targets']
video_train, video_test, target_train, target_test = train_test_split(
    landmarks, 
    target, 
    test_size=0.15, #по 3 видео для проверки на каждый класс
    random_state=42,
    stratify=target
)